# Run Inference on Blind Dataset

- Choose preprocessing + model pipeline to use for inference on blind dataset based on model performance
- Load model from saved models 
- Run inference
- Save predictions

In [ ]:
# Importing libraries
from pathlib import Path
import sys
import numpy as np

In [ ]:
# To be able to access src
sys.path.append(str(Path().resolve().parent))

In [ ]:
# Reloading files if changes have been made
import importlib
import src.preprocessing
import src.models
import src.utils
import src.train
# import src.evaluate
import src.visualisation
import src.trainDL
# import src.evaluateDL
import src.inference
importlib.reload(src.preprocessing)
importlib.reload(src.models)
importlib.reload(src.utils)
# importlib.reload(src.evaluate)
importlib.reload(src.train)
importlib.reload(src.visualisation)
# importlib.reload(src.evaluateDL)
importlib.reload(src.trainDL)
importlib.reload(src.inference)

In [ ]:
# Load blind dataset, and clean the data
from src.preprocessing import load_blind_dataset, clean_data
blind_dataset_path = Path("..") / "pur-1-data-main" / "data" / "Task 1" / "Blind Dataset" 
blind_dfs = load_blind_dataset(blind_dataset_path)
blind_dfs_clean = clean_data(blind_dfs) # 15 x 800 x 12 (no target column)

In [ ]:
# Load all models
models_dir = Path("models") / "final_models"

from src.inference import load_all_models
loaded_models = load_all_models(models_dir)

In [ ]:
loaded_models['LSTMClassifier_raw_data']

In [ ]:
# Prepare the input data for inference
feature_cols = ['Channel 4 flux', 'Downstream conductivity', 'Downstream water temperature', 'SS1 state', 'SS1 position', 'SS2 state', 'SS2 position', 'Upstream water temperature']
lstm_feature_cols = ['Channel 4 flux', 'SS1 position', 'SS2 position']

from src.inference import prepare_inputs
X_data = prepare_inputs(blind_dfs_clean, feature_cols, lstm_feature_cols) # dictionary with keys for preprocessing 

In [ ]:
# Run inference
from src.inference import run_inference
predictions = run_inference(loaded_models, X_data)

In [ ]:
# Heatmap
from src.inference import plot_vote_matrix_heatmap
plot_vote_matrix_heatmap(predictions, sort_by_disagreement=False)

In [ ]:
from src.inference import majority_vote
final_predictions = majority_vote(predictions)
print(final_predictions)

In [ ]:
i=0
for df in blind_dfs_clean:
    print(df['ID'].iloc[0])
    print(final_predictions[i])
    i += 1

In [ ]:
# print event id and predicted label for each event
for event_id, label in final_predictions.items():
    print(f"Event ID: {event_id}, Predicted Label: {label}")

### Look at events the models disagree on

In [ ]:
from src.inference import get_disagreement_IDs
disagreement_ids = get_disagreement_IDs(predictions, blind_dfs_clean)
disagreement_ids = ["133722", "30569"] # IDs that more than just LSTM disagreed on (should be removed once LSTM is fixed)

In [ ]:
from src.preprocessing import combine_csvs
combined_blind_df = combine_csvs(blind_dfs_clean)

In [ ]:
from src.visualisation import plot_selected_events_grid
plot_selected_events_grid(
    combined_blind_df,
    feature_cols,
    IDs=disagreement_ids,
)

### Plot blind dataset (raw timeseries)

In [ ]:
from src.visualisation import plot_raw_timeseries
plot_raw_timeseries(combined_blind_df, feature_cols, target_col=None, id_col="ID", time_col="timestamp", class_value=None, title="Blind Dataset Raw Data", ncols=2)

In [ ]:
# Outliers
from src.utils import top_outliers
outlier_df = top_outliers(combined_blind_df, feature_cols)

In [ ]:
from src.visualisation import top_outlier_plots
top_outlier_plots(combined_blind_df, outlier_df, feature_cols, top_n=8)

# Hmm don't know what I want to say about these "outliers"
# Maybe I can have a look at the ones my models disagree on 
# before drawing any conclusions

In [ ]:
# Have a look at the flatlines:

In [ ]:
columns_for_comp = blind_dfs_clean[0].columns.drop("ID").drop("timestamp").to_list()

In [ ]:
# Have a look at the flatlines
ID_flatline = '30569'
from src.visualisation import plot
plot(combined_blind_df, feature_cols=columns_for_comp, ID=ID_flatline)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_selected_events(
    combined_df,
    feature_cols,
    IDs,
):

    # Ensure list
    if not isinstance(IDs, (list, tuple, set, np.ndarray)):
        IDs = [IDs]

    for col in feature_cols:

        # Create separate figure for each feature
        plt.figure(figsize=(8, 6))

        # -----------------------------------
        # Background trajectories
        # -----------------------------------
        for event_id in combined_df["ID"].unique():

            subset = (
                combined_df[combined_df["ID"] == event_id]
                .sort_values("timestamp")
            )

            plt.plot(
                subset["timestamp"],
                subset[col],
                color="gray",
                alpha=0.15,
                linewidth=2,
            )

        # -----------------------------------
        # Median trajectory
        # -----------------------------------
        median_series = (
            combined_df
            .groupby("timestamp")[col]
            .median()
        )

        plt.plot(
            median_series.index,
            median_series.values,
            color="black",
            linewidth=3,
            linestyle="--",
            label="Median",
        )

        # -----------------------------------
        # Selected IDs
        # -----------------------------------
        for event_id in IDs:

            subset = (
                combined_df[combined_df["ID"] == event_id]
                .sort_values("timestamp")
            )

            if len(subset) == 0:
                continue

            plt.plot(
                subset["timestamp"],
                subset[col],
                linewidth=2.5,
                label=f"ID {event_id}",
                color='darkorange'
            )

        plt.title(col)
        plt.xlabel("Timestamp")
        plt.ylabel(col)
        plt.yscale("log")
        plt.grid(True, alpha=0.2)
        plt.legend()
        plt.tight_layout()
        plt.show()

plot_selected_events(
    combined_blind_df,
    feature_cols,
    IDs=disagreement_ids[1],
)

In [ ]:
loaded_models['DecisionTree_Flattened_features']

In [ ]:
# Have a look at the tree in the decision tree model

dec_tree_flattened = loaded_models['DecisionTree_Flattened_features']
dec_tree_extracted = loaded_models['DecisionTree_Extracted_features']

dec_tree_flattened['model']

In [ ]:
from sklearn import tree
import matplotlib.pyplot as plt

# Extract the trained model
model = dec_tree_flattened['model']
model = dec_tree_extracted['model']

plt.figure(figsize=(20,10))
tree.plot_tree(
    model,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()

In [ ]:
model = dec_tree_flattened['model']

plt.figure(figsize=(20,10))
tree.plot_tree(
    model,
    feature_names=model.feature_names_in_,  # list of feature names
    class_names=['Gang Lower', 'SCRAM'], 
    filled=True,
    rounded=True,
    fontsize=17)
plt.show()

model = dec_tree_extracted['model']

plt.figure(figsize=(20,10))
tree.plot_tree(
    model,
    feature_names=model.feature_names_in_,  # list of feature names
    class_names=['Gang Lower', 'SCRAM'], 
    filled=True,
    rounded=True,
    fontsize=17)
plt.show()